# مدير ملفاتي — Telegram إلى Google Drive
افتح الدفتر في Google Colab واضغط تشغيل في الخلية الوحيدة. في أول استخدام، أضف `TELEGRAM_BOT_TOKEN` و`OWNER_ID` في لوحة **Secrets** واسمح للدفتر بالوصول إليهما. لا تشارك الدفتر أو الأسرار.

In [ ]:
# واجهة التشغيل اليومية — شغّل هذه الخلية فقط.
# إعداد الروابط يتم من الواجهة الظاهرة أسفل الخلية؛ لا تعدّل الكود.

import json
import os
import re
import subprocess
import sys
import threading
import uuid
from pathlib import Path

from IPython.display import display, Markdown
import ipywidgets as widgets

ENGINE_COMMIT = "40c60ec9ca85f4a74d69661ace187ca6ab9c7d61"
REPO_URL = "https://github.com/Hus-GPT/TelegramDriveBot.git"
REPO_DIR = Path("/content/TelegramDriveBot")
DRIVE_DESTINATION = Path("/content/drive/MyDrive/TelegramDriveBot")
INBOX_DIR = DRIVE_DESTINATION / ".state" / "colab_link_inbox"

print("مرحباً بك في مدير ملفات Google Drive.")
print("أولاً: التحقق من الإعدادات الآمنة...")

try:
    from google.colab import drive, userdata
    bot_token = userdata.get("TELEGRAM_BOT_TOKEN")
    owner_id = userdata.get("OWNER_ID")
except Exception:
    bot_token, owner_id = "", ""

if not bot_token or not owner_id or not str(owner_id).isdigit():
    print("\nيلزم إعداد سرَّين مرة واحدة في لوحة Secrets في Colab:")
    print("1) TELEGRAM_BOT_TOKEN — رمز البوت من BotFather.")
    print("2) OWNER_ID — رقم حسابك في Telegram.")
    print("فعّل السماح للدفتر بالوصول إلى السرّين، ثم أعد تشغيل الخلية.")
    raise RuntimeError("الإعدادات المطلوبة غير مكتملة.")

os.environ["TELEGRAM_BOT_TOKEN"] = str(bot_token)
os.environ["OWNER_ID"] = str(owner_id)

print("ثانياً: ربط Google Drive...")
drive.mount("/content/drive")
INBOX_DIR.mkdir(parents=True, exist_ok=True)

print("ثالثاً: تجهيز نسخة المحرك المعتمدة...")
if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR)],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", ENGINE_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", "--detach", ENGINE_COMMIT], check=True)

print("رابعاً: تجهيز المكتبات...")
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    capture_output=True,
    text=True,
)
if install.returncode:
    print("تعذّر تجهيز المكتبات. أعد تشغيل الخلية، وإذا تكرر الخطأ أرسل رسالة الخطأ الظاهرة.")
    raise RuntimeError("Dependency setup failed.")

# Build the link-paste UI. Each submitted URL is written as a separate atomic inbox file.
title = widgets.HTML("<h3>📥 مدير ملفات Google Drive</h3><p>الصق رابط تنزيل مباشر، أو عدة روابط (كل رابط في سطر).</p>")
links_box = widgets.Textarea(
    value="",
    placeholder="https://example.com/file.zip\nhttps://example.com/document.pdf",
    description="الروابط:",
    layout=widgets.Layout(width="100%", height="130px"),
    style={"description_width": "initial"},
)
submit_button = widgets.Button(description="إضافة الروابط إلى قائمة التنزيل", button_style="primary", icon="plus")
status = widgets.HTML("<span>الحالة: جاهز لاستقبال الروابط بعد بدء البوت.</span>")
ui = widgets.VBox([title, links_box, submit_button, status], layout=widgets.Layout(width="100%"))
display(ui)

def submit_links(_):
    raw = links_box.value or ""
    candidates = [line.strip() for line in raw.splitlines() if line.strip()]
    if not candidates:
        status.value = "<span style='color:#b45309'>الصق رابطاً واحداً على الأقل.</span>"
        return
    if len(candidates) > 20:
        status.value = "<span style='color:#b45309'>أرسل 20 رابطاً كحد أقصى في الدفعة الواحدة.</span>"
        return
    invalid = [url for url in candidates if not re.match(r"^https?://[^\\s]+$", url, re.IGNORECASE)]
    if invalid:
        status.value = "<span style='color:#b91c1c'>تأكد أن كل سطر رابط يبدأ بـ http:// أو https://.</span>"
        return
    try:
        for url in candidates:
            task_id = uuid.uuid4().hex
            temp_path = INBOX_DIR / (task_id + ".tmp")
            final_path = INBOX_DIR / (task_id + ".json")
            with temp_path.open("w", encoding="utf-8") as stream:
                json.dump({"url": url}, stream, ensure_ascii=False)
            os.replace(temp_path, final_path)
        links_box.value = ""
        status.value = "<span style='color:#15803d'>تم إرسال " + str(len(candidates)) + " رابطاً إلى طابور المعالجة. تابع التقدم من Telegram عبر /status.</span>"
    except Exception:
        status.value = "<span style='color:#b91c1c'>تعذرت إضافة الروابط. تأكد أن Google Drive ما زال متصلاً.</span>"

submit_button.on_click(submit_links)

print("خامساً: بدء البوت في الخلفية...")
print("اترك جلسة Colab تعمل. استخدم لوحة الروابط أعلاه، أو أرسل الروابط والملفات إلى البوت في Telegram.")
process = subprocess.Popen(
    [sys.executable, "-u", "-m", "app.colab_start"],
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

def stream_safe_logs():
    for line in process.stdout:
        safe_line = line.replace(str(bot_token), "[محجوب]")
        safe_line = safe_line.replace(str(owner_id), "[محجوب]")
        if "ERROR" in safe_line or "WARNING" in safe_line or "Traceback" in safe_line or "started" in safe_line.lower():
            print(safe_line.rstrip())

threading.Thread(target=stream_safe_logs, daemon=True).start()
print("ظهرت واجهة لصق الروابط. لا توقف الخلية أثناء التنزيل.")
